# feral — LU engine, factor access & introspection

The 0.11.0 Python surface is **purely additive**: the original `Solver`/`CscMatrix` workflow is unchanged, and these features sit alongside it. This notebook covers three additions:

1. the unsymmetric **LU basis engine** (`LuFactor` / `LuMatrix`),
2. **factor access** — the assembled `L` and `D`, and the symbolic analysis, and
3. **introspection** — tuning knobs, pivot magnitudes, factor stats, and scaling info.

Every cell carries its own external oracle (a dense `numpy` solve or a reconstruction identity), so the notebook self-checks when executed.

In [1]:
import numpy as np
import feral

print('feral', feral.__version__)

feral 0.13.0


## 1. Unsymmetric LU basis engine

`LuFactor` factors a **general** (unsymmetric) square matrix and solves `A x = b` with `ftran` and `Aᵀ y = c` with `btran`. It auto-routes between a dense and a sparse engine; the oracle here is `numpy.linalg.solve`.

In [2]:
A = np.array([[2.0, 1.0, 0.0],
              [0.0, 3.0, 1.0],
              [1.0, 0.0, 4.0]])
lu = feral.LuFactor(feral.LuMatrix.from_dense(A))
print('dense engine:', lu.is_dense)

b = np.array([1.0, 2.0, 3.0])
x = lu.ftran(b)                       # solve A x = b
print('ftran residual:', f'{np.max(np.abs(A @ x - b)):.2e}')
assert np.allclose(x, np.linalg.solve(A, b))

c = np.array([1.0, 0.0, 0.0])
y = lu.btran(c)                       # solve Aᵀ y = c
assert np.allclose(y, np.linalg.solve(A.T, c))

dense engine: True
ftran residual: 0.00e+00


### Factor identity and product-form updates

The returned permutations satisfy `P A Q = L U`. A simplex-style `update` replaces one basis column in product form; `refactor` rebuilds from scratch and resets the update counter.

In [3]:
L = lu.l_array()
U = lu.u_array()
lhs = A[np.ix_(lu.perm, lu.qcol)]     # P A Q
print('|P A Q - L U|inf:', f'{np.max(np.abs(lhs - L @ U)):.2e}')
assert np.allclose(lhs, L @ U)

new_col = np.array([0.0, 5.0, 1.0])
lu.update(1, new_col)                 # replace basis column 1
A2 = A.copy(); A2[:, 1] = new_col
print('updates_since_refactor:', lu.updates_since_refactor)
assert np.allclose(lu.ftran(b), np.linalg.solve(A2, b))

lu.refactor(feral.LuMatrix.from_dense(A))
assert lu.updates_since_refactor == 0

|P A Q - L U|inf: 2.78e-17
updates_since_refactor: 1


A singular basis raises `SingularBasisError` — a subclass of `FactorError`, so existing `except FactorError` handlers keep working.

In [4]:
singular = np.array([[1.0, 2.0], [2.0, 4.0]])   # rank 1
try:
    feral.LuFactor(feral.LuMatrix.from_dense(singular), force_dense=True)
except feral.SingularBasisError as e:
    print('caught SingularBasisError:', e)
    assert isinstance(e, feral.FactorError)

caught SingularBasisError: LU basis is singular at column 1


## 2. Factor access — L, D, and the symbolic structure

After a symmetric `Solver.factor`, `Solver.factors()` exposes the assembled unit-lower `L` and block-diagonal `D` in **factorization order**. The reconstruction identity is the oracle:

$$ L\,D\,L^{\top} = P\,(S A S)\,P^{\top} $$

with the permutation `fac.perm` and the per-row scaling `fac.scaling`.

In [5]:
M = np.array([
    [2.0, 1.0, 0.0, 0.0],
    [1.0, -3.0, 1.0, 0.0],
    [0.0, 1.0, 4.0, 1.0],
    [0.0, 0.0, 1.0, -2.0],
])
csc = feral.CscMatrix.from_dense(M)
s = feral.Solver()
status, inertia = s.factor(csc)
print('inertia:', inertia)

fac = s.factors()
indptr, indices, data = fac.l_csc()
d_diag, d_sub = fac.d_blocks()

inertia: Inertia(n_pos=2, n_neg=2, n_zero=0)


In [6]:
n = fac.n
L = np.zeros((n, n))
for j in range(n):
    for k in range(indptr[j], indptr[j + 1]):
        L[indices[k], j] = data[k]

D = np.zeros((n, n))
i = 0
while i < n:
    if i + 1 < n and d_sub[i] != 0.0:        # 2x2 pivot block
        D[i, i] = d_diag[i]; D[i + 1, i + 1] = d_diag[i + 1]
        D[i, i + 1] = D[i + 1, i] = d_sub[i]
        i += 2
    else:
        D[i, i] = d_diag[i]; i += 1

perm, sc = fac.perm, fac.scaling
lhs = L @ D @ L.T
rhs = np.array([[sc[perm[a]] * M[perm[a], perm[b]] * sc[perm[b]]
                 for b in range(n)] for a in range(n)])
print('|L D Lᵀ - P(SAS)Pᵀ|inf:', f'{np.max(np.abs(lhs - rhs)):.2e}')
assert np.allclose(lhs, rhs)

|L D Lᵀ - P(SAS)Pᵀ|inf: 5.55e-17


### Symbolic analysis without a numeric factor

`feral.analyze` runs the ordering + symbolic factorization with **no** numeric work. On a larger system the predicted `factor_nnz_estimate` is a (slack-inflated) upper bound on the realized factor nnz — provided the numeric factor uses the **same ordering** as the analysis.

In [7]:
rng = np.random.default_rng(2)
B = rng.standard_normal((15, 15))
spd = feral.CscMatrix.from_dense(B @ B.T + 15 * np.eye(15))

sym = feral.analyze(spd, ordering='amd')
snum = feral.Solver(ordering='amd')
snum.factor(spd)

print('ordering        :', sym.ordering)
print('num_supernodes  :', sym.num_supernodes)
print('nnz estimate    :', sym.factor_nnz_estimate)
print('realized nnz    :', snum.factor_nnz)
assert sorted(sym.perm) == list(range(sym.n))      # perm is a permutation
assert sym.factor_nnz_estimate >= snum.factor_nnz

ordering        : amd
num_supernodes  : 1
nnz estimate    : 144
realized nnz    : 120


## 3. Introspection — knobs, pivots, stats, scaling

Different fill-reducing orderings must agree on the certified inertia. Turning on `profiling` populates a `ProfileReport`; `last_factor_stats` and the pivot-magnitude getters summarize the numeric factor.

In [8]:
rng = np.random.default_rng(0)
G = rng.standard_normal((14, 14)); G = G + G.T
G += np.diag(rng.standard_normal(14))
Gc = feral.CscMatrix.from_dense(G)

inertias = set()
for ordering in ('auto', 'amd', 'amf'):
    st = feral.Solver(ordering=ordering)
    _, inrt = st.factor(Gc)
    inertias.add(inrt.as_tuple())
print('distinct inertias across orderings:', inertias)
assert len(inertias) == 1                          # ordering-invariant

distinct inertias across orderings: {(6, 8, 0)}


In [9]:
sp = feral.Solver(profiling=True)
sp.factor(Gc)
fs = sp.last_factor_stats()
print('nnz_a / nnz_l :', fs.nnz_a, '/', fs.nnz_l)
print('fill_ratio    :', f'{fs.fill_ratio:.2f}')
print('pivot range   :', f'{sp.min_pivot_magnitude:.2e}',
      '..', f'{sp.max_pivot_magnitude:.2e}')
print('scaling kind  :', sp.scaling_info.kind)
print('profile total :', sp.profile_report().total_us, 'us')
assert fs.fill_ratio >= 1.0
assert 0.0 < sp.min_pivot_magnitude <= sp.max_pivot_magnitude

nnz_a / nnz_l : 105 / 105
fill_ratio    : 1.00
pivot range   : 7.28e-01 .. 3.67e+00
scaling kind  : applied
profile total : 0 us
